In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("/content/mobile_recommendation_system_dataset.csv")
df.head()

,name,ratings,price,imgURL,corpus
0,"REDMI Note 12 Pro 5G (Onyx Black, 128 GB)",4.2,23999,https://rukminim2.flixcart.com/image/312/312/x...,Storage128 GBRAM6 SystemAndroid 12Processor T...
1,"OPPO F11 Pro (Aurora Green, 128 GB)",4.5,"₹20,999",https://rukminim2.flixcart.com/image/312/312/k...,Storage128 GBRAM6 GBExpandable Storage256GB S...
2,"REDMI Note 11 (Starburst White, 64 GB)",4.2,13149,https://rukminim2.flixcart.com/image/312/312/x...,Storage64 GBRAM4 SystemAndroid 11Processor Sp...
3,"OnePlus Nord CE 5G (Blue Void, 256 GB)",4.1,21999,https://rukminim2.flixcart.com/image/312/312/x...,Storage256 GBRAM12 SystemAndroid Q 11Processo...
4,"APPLE iPhone 13 mini (Blue, 128 GB)",4.6,3537,https://rukminim2.flixcart.com/image/312/312/k...,Storage128 SystemiOS 15Processor TypeA15 Bion...


In [2]:
def clean_text_fixed(row):
    name = row['name'] if pd.notna(row['name']) else ""
    corpus = row['corpus'] if pd.notna(row['corpus']) else ""
    return f"{name} {corpus}".lower()

In [3]:
from sentence_transformers import SentenceTransformer

df['text'] = df.apply(clean_text_fixed, axis=1)

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
df['embeddings'] = df['text'].apply(lambda x: model.encode(x))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
df.to_pickle('product_embeddings.pkl')
df = pd.read_pickle('product_embeddings.pkl')

In [5]:
def recommend_products(query, top_k=5):
  query = query.lower()
  query_embedding = model.encode(query)
  df['similarity'] = df['embeddings'].apply(lambda x: cosine_similarity([query_embedding], [x]).flatten()[0])

  recommendations = df.sort_values(by='similarity', ascending=False).head(top_k)
  return recommendations[['name', 'price', 'ratings', 'similarity']]

query = "8GB RAM smartphone"
recommendations = recommend_products(query)
recommendations

,name,price,ratings,similarity
2233,"SAMSUNG Galaxy Note 8 (Orchid Grey, 64 GB)",8464,4.2,0.582687
649,"SAMSUNG Galaxy Note 8 (Midnight Black, 64 GB)",8464,4.1,0.575937
2180,"SAMSUNG Galaxy J8 (Blue, 64 GB)",17000,4.4,0.566143
353,"SAMSUNG Galaxy S8 (Midnight Black, 64 GB)",15546,4.3,0.566103
873,"SAMSUNG Galaxy J8 (Black, 64 GB)",17000,4.2,0.558446


In [6]:
query = "Oppo Smart phone"
recommendations = recommend_products(query)
recommendations

,name,price,ratings,similarity
1715,"OPPO F11 Pro (Thunder Black, 128 GB)","₹29,990",4.5,0.663767
1186,"OPPO F11 Pro (Thunder Black, 64 GB)","₹18,500",4.3,0.659924
1,"OPPO F11 Pro (Aurora Green, 128 GB)","₹20,999",4.5,0.655567
1886,"OPPO F11 Pro (Waterfall Grey, 128 GB)","₹29,990",4.5,0.644885
785,"OPPO A 78 5G (Glowing Black, 128 GB)","₹18,690",4.0,0.644465
